# Stage 8.6 -- Generic Map Area Discovery

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

BASE = Path('../data/gold/maps/area_discovery')
MAP_ID = 'inferno'

def load(name):
    return pd.read_parquet(BASE / f'{name}.parquet')

summary = load('map_area_discovery_summary')
inventory = load('map_place_inventory')
coverage = load('map_place_coverage')
coordinates = load('map_place_coordinates')
sample = load('map_place_coordinate_sample')
stability = load('map_place_name_stability')
vertical = load('map_place_vertical_profile')
unknowns = load('map_area_discovery_unknowns')
crosswalk = load('mirage_place_registry_crosswalk')
audit = load('map_area_discovery_audit')

def scoped(df):
    return df[df['map_id'].eq(MAP_ID)].copy() if 'map_id' in df.columns else df.iloc[0:0].copy()

## Discovery Summary

In [ ]:
display(scoped(summary))

## Place Frequency

In [ ]:
freq = scoped(inventory).sort_values('tick_count', ascending=False)
display(freq)
ax = freq.head(25).plot.bar(x='raw_place', y='tick_count', figsize=(12, 4), legend=False)
ax.set_ylabel('ticks')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()

## Demo Coverage

In [ ]:
cov = scoped(coverage).sort_values('demo_coverage_share', ascending=False)
display(cov)
ax = cov.plot.bar(x='raw_place', y='demo_coverage_share', figsize=(12, 4), legend=False)
ax.set_ylim(0, 1.05)
plt.xticks(rotation=60, ha='right')
plt.tight_layout()

## Round Coverage

In [ ]:
display(cov[['raw_place', 'rounds_with_place', 'round_coverage_share', 'coverage_status']])

## Coordinate Profile

In [ ]:
display(scoped(coordinates))

## X/Y Coordinate Scatter

In [ ]:
pts = scoped(sample)
fig, ax = plt.subplots(figsize=(7, 7))
for place, group in pts.groupby('raw_place'):
    ax.scatter(group['X'], group['Y'], s=4, alpha=0.25)
centers = scoped(coordinates)
for _, row in centers.iterrows():
    ax.text(row['x_median'], row['y_median'], row['raw_place'], fontsize=8)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_aspect('equal', adjustable='box')
plt.tight_layout()

## Median X/Y by Place

In [ ]:
centers = scoped(coordinates)
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(centers['x_median'], centers['y_median'])
for _, row in centers.iterrows():
    ax.text(row['x_median'], row['y_median'], row['raw_place'], fontsize=8)
ax.set_xlabel('median X')
ax.set_ylabel('median Y')
ax.set_aspect('equal', adjustable='box')

## Z Profile

In [ ]:
z = scoped(vertical).sort_values('z_range', ascending=False)
display(z)
ax = z.plot.bar(x='raw_place', y='z_range', figsize=(12, 4), legend=False)
ax.set_ylabel('Z range')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()

## Place Stability

In [ ]:
display(scoped(stability))

## Unknowns

In [ ]:
display(scoped(unknowns))

## Mirage Registry Crosswalk

In [ ]:
display(scoped(crosswalk) if MAP_ID == 'mirage' else 'Crosswalk is only available for Mirage in this stage.')

## Final Readiness

In [ ]:
display(scoped(audit))